In [ ]:
from pymycobot.mycobot import MyCobot
import numpy as np
import matplotlib.pyplot as plt
import time

#Conexion al robot
mc = MyCobot('/dev/ttyTHS1', 115200) 
mc.power_on()
time.sleep(1)

if not mc.is_controller_connected():
    print("No se pudo conectar al robot, verificar puerto serie")
    exit()

print("Robot conectado de forma local")

# Pose segura antes de iniciar
mc.send_angles([0, 0, 0, 0, 0, -45], 20)
time.sleep(3)
print("Pose inicial OK")


#parametros del robot sacados de la tabla DH del examen

DH_PARAMS = [
    (0.0,   131.56,  90),
    (110.4,   0.0,    0),
    (96.0,    0.0,    0),
    (0.0,    66.39,  -90),
    (0.0,    73.18,   90),
    (0.0,    48.6,    0),
]

L2 = 110.4
L3 = 96.0
d1 = 131.56

JOINT_LIMITS = {
    'j1': (-168, 168),
    'j2': (-135,  90),
    'j3': (-150, 150),
    'j4': (-145, 145),
    'j5': (-165, 165),
    'j6': (-180, 180),
}


# FK — necesaria para la verificacion matematica al final

def matriz_dh(a, d, alpha, theta):
    alpha_r = np.radians(alpha)
    return np.array([
        [np.cos(theta),
         -np.sin(theta)*np.cos(alpha_r),
          np.sin(theta)*np.sin(alpha_r),
          a*np.cos(theta)],
        [np.sin(theta),
          np.cos(theta)*np.cos(alpha_r),
         -np.cos(theta)*np.sin(alpha_r),
          a*np.sin(theta)],
        [0, np.sin(alpha_r), np.cos(alpha_r), d],
        [0, 0, 0, 1]
    ])


class ForwardKinematics:

    def __init__(self, dh_params):
        self.dh = dh_params

    def calcular(self, angulos_deg):
        angulos_rad = np.radians(angulos_deg)
        T = np.eye(4)
        for i in range(6):
            a, d, alpha = self.dh[i]
            Ti = matriz_dh(a, d, alpha, angulos_rad[i])
            T = T @ Ti
        pos = T[:3, 3]
        return T, pos


# IK analitica — formulas de la pagina 7 del examen
# theta1 = atan2(y, x)
# cos(theta3) = (r^2 + z'^2 - L2^2 - L3^2) / (2*L2*L3)
# theta2 = atan2(z', r) - atan2(L3*sin3, L2 + L3*cos3)

class InverseKinematics:

    def __init__(self):
        self.L2 = L2
        self.L3 = L3
        self.d1 = d1

    def calcular(self, x, y, z):
        # rotacion de la base hacia el objetivo
        theta1 = np.degrees(np.arctan2(y, x))

        r       = np.sqrt(x**2 + y**2)
        z_prima = z - self.d1

        # verificar si el punto es alcanzable
        distancia   = np.sqrt(r**2 + z_prima**2)
        alcance_max = self.L2 + self.L3

        if distancia > alcance_max:
            print(f"  ({x},{y},{z}) queda fuera del espacio de trabajo")
            print(f"  distancia necesaria {distancia:.1f}mm, "
                  f"maximo {alcance_max:.1f}mm")
            return None

        #angulo del codo por ley del coseno
        cos_t3 = (r**2 + z_prima**2 - self.L2**2 - self.L3**2) \
                 / (2 * self.L2 * self.L3)
        cos_t3 = np.clip(cos_t3, -1, 1)

        #usamos codo arriba como configuracion principal
        sin_t3 = np.sqrt(1 - cos_t3**2)
        theta3 = np.degrees(np.arctan2(sin_t3, cos_t3))

        # angulo del hombro
        theta2 = np.degrees(
            np.arctan2(z_prima, r) -
            np.arctan2(
                self.L3 * sin_t3,
                self.L2 + self.L3 * cos_t3
            )
        )

        angles = [
            round(theta1, 2),
            round(theta2, 2),
            round(theta3, 2),
            0.0, 0.0, 0.0
        ]

        # revisar que ningun angulo se pase de los limites fisicos
        nombres = ['j1','j2','j3','j4','j5','j6']
        for i, (ang, nom) in enumerate(zip(angles, nombres)):
            lim_min, lim_max = JOINT_LIMITS[nom]
            if not (lim_min <= ang <= lim_max):
                print(f"  advertencia: theta{i+1}={ang:.2f} grados "
                      f"fuera de [{lim_min}, {lim_max}]")

        return angles


def ik_solve(x, y, z):
    # funcion que usa el rol de control (A) en control.py
    ik = InverseKinematics()
    return ik.calcular(x, y, z)


# ----------------------------------------------------------
# Posiciones de prueba para la tabla del informe
# elegidas para cubrir distintas zonas del espacio de trabajo
# ----------------------------------------------------------
POSICIONES_IK = [
    {"id": 1, "x": 150, "y":   0, "z": 200},
    {"id": 2, "x": 200, "y": -50, "z": 180},
    {"id": 3, "x": 100, "y":  50, "z": 250},
]

ik         = InverseKinematics()
fk         = ForwardKinematics(DH_PARAMS)
resultados = []

# ----------------------------------------------------------
# Tabla de comparacion IK analitica vs robot real
# ----------------------------------------------------------
print()
print("=" * 70)
print("  TABLA P3 — IK Analitica vs IK Robot Real")
print("=" * 70)

for pos in POSICIONES_IK:
    x, y, z = pos["x"], pos["y"], pos["z"]
    pid     = pos["id"]

    print(f"\n  Posicion {pid}: ({x}, {y}, {z}) mm")
    print(f"  {'-'*60}")

    # calcular con nuestras formulas
    analitica = ik.calcular(x, y, z)

    if analitica is None:
        print("  sin solucion analitica para este punto")
        continue

    # mandar al robot y leer lo que el resolvio internamente
    mc.send_coords([x, y, z, -90, -45, -90], 20, 1)
    time.sleep(4)
    robot = mc.get_angles()
    robot = [round(a, 2) for a in robot]

    # calcular diferencia en grados
    e1 = abs(analitica[0] - robot[0])
    e2 = abs(analitica[1] - robot[1])
    e3 = abs(analitica[2] - robot[2])
    e_prom = round((e1 + e2 + e3) / 3, 2)

    print(f"  {'':10} {'J1':>9} {'J2':>9} {'J3':>9}")
    print(f"  {'Analitica':10} "
          f"{analitica[0]:>9.2f} "
          f"{analitica[1]:>9.2f} "
          f"{analitica[2]:>9.2f}")
    print(f"  {'Robot':10} "
          f"{robot[0]:>9.2f} "
          f"{robot[1]:>9.2f} "
          f"{robot[2]:>9.2f}")
    print(f"  {'Error(°)':10} "
          f"{e1:>9.2f} "
          f"{e2:>9.2f} "
          f"{e3:>9.2f}  "
          f"promedio={e_prom}°")

    resultados.append({
        "id":        pid,
        "objetivo":  [x, y, z],
        "analitica": analitica,
        "robot":     robot,
        "errores":   [e1, e2, e3],
        "e_prom":    e_prom
    })

    # volver a pose segura entre cada prueba
    mc.send_angles([0, 0, 0, 0, 0, -45], 20)
    time.sleep(3)

if resultados:
    e_general = round(
        sum(r["e_prom"] for r in resultados) / len(resultados), 2
    )
    print(f"\n  error promedio general: {e_general} grados")

print("\n" + "=" * 70)


# Singularidades identificadas en el espacio de trabajo

print()
print("=" * 70)
print("  SINGULARIDADES")
print("=" * 70)

casos = [
    {
        "tipo" : "singularidad de codo",
        "desc" : "brazo totalmente extendido",
        "x": 206, "y": 0, "z": 131,
        "causa": "cos(theta3) tiende a 1, el robot pierde "
                 "un grado de libertad"
    },
    {
        "tipo" : "singularidad de hombro",
        "desc" : "directamente encima de la base",
        "x": 0, "y": 0, "z": 280,
        "causa": "atan2(0,0) queda indefinido, "
                 "theta1 puede ser cualquier valor"
    },
    {
        "tipo" : "fuera de alcance",
        "desc" : "posicion imposible de alcanzar",
        "x": 300, "y": 0, "z": 300,
        "causa": "la distancia al objetivo supera "
                 "L2+L3=206.4mm"
    },
]

for caso in casos:
    print(f"\n  tipo    : {caso['tipo']}")
    print(f"  caso    : {caso['desc']}")
    print(f"  pos     : ({caso['x']}, {caso['y']}, {caso['z']}) mm")
    print(f"  causa   : {caso['causa']}")
    r = ik.calcular(caso['x'], caso['y'], caso['z'])
    if r:
        print(f"  resultado: solucion encontrada — "
              f"theta1={r[0]:.2f}  "
              f"theta2={r[1]:.2f}  "
              f"theta3={r[2]:.2f}")
    else:
        print(f"  resultado: sin solucion — singularidad confirmada")

print("\n" + "=" * 70)


# Grafica del espacio de trabajo en el plano XZ

print()
print("Generando grafica del espacio de trabajo...")

px, pz = [], []
for x in range(0, 215, 4):
    for z in range(0, 350, 4):
        if ik.calcular(x, 0, z) is not None:
            px.append(x)
            pz.append(z)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(px, pz, s=6, c='steelblue',
           alpha=0.3, label='zona alcanzable')

colores = ['red', 'green', 'orange']
for i, pos in enumerate(POSICIONES_IK):
    ax.scatter(pos["x"], pos["z"],
               s=150, c=colores[i], zorder=5,
               label=f'P{pos["id"]} ({pos["x"]},{pos["z"]})')
    ax.annotate(f'  P{pos["id"]}',
                (pos["x"], pos["z"]),
                fontsize=11, fontweight='bold')

ax.set_xlabel("X (mm)", fontsize=12)
ax.set_ylabel("Z (mm)", fontsize=12)
ax.set_title("Espacio de trabajo MyCobot 280 — plano XZ",
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-10, 230)
ax.set_ylim(-10, 360)
plt.tight_layout()
plt.savefig("espacio_trabajo_p3.png", dpi=150)
plt.show()
print("grafica guardada como espacio_trabajo_p3.png")


# Verificacion matematica IK → FK
# comprobamos que los angulos calculados recuperan
# la posicion original cuando los pasamos por FK

print()
print("=" * 70)
print("  VERIFICACION IK → FK")
print("=" * 70)

for r in resultados:
    x, y, z   = r["objetivo"]
    analitica = r["analitica"]

    _, pos_fk = fk.calcular(analitica)
    ex    = abs(x - pos_fk[0])
    ey    = abs(y - pos_fk[1])
    ez    = abs(z - pos_fk[2])
    e_pos = np.sqrt(ex**2 + ey**2 + ez**2)

    print(f"\n  posicion {r['id']}: objetivo ({x},{y},{z}) mm")
    print(f"  FK recupero : "
          f"[{pos_fk[0]:.2f}, {pos_fk[1]:.2f}, {pos_fk[2]:.2f}] mm")
    print(f"  error       : {e_pos:.4f} mm")
    print(f"  estado      : "
          f"{'ok' if e_pos < 1.0 else 'revisar parametros DH'}")

print("\n" + "=" * 70)

# volver a pose segura al terminar todo
mc.send_angles([0, 0, 0, 0, 0, -45], 20)
time.sleep(3)
print()
print("P3 completado")
print("robot en pose segura")